# Day 38 – Q1: Word2Vec on ShopSense Reviews

Train Word2Vec, demonstrate polysemy of 'cheap', build context-based disambiguation, and compare window_size=2 vs 10.

In [1]:
import re
import numpy as np
import pandas as pd
from gensim.models import Word2Vec
from sklearn.metrics.pairwise import cosine_similarity

REVIEWS_PATH = "shopsense_reviews.csv"
TARGET_WORD  = "cheap"
ANCHOR_AFFORDABLE    = ["affordable", "value", "budget", "inexpensive", "price"]
ANCHOR_LOW_QUALITY   = ["flimsy", "poor", "terrible", "cheap", "bad", "worthless"]
WINDOW_SIZES = [2, 10]


c:\Users\SHREE\miniconda3\envs\mgenv\lib\site-packages\google\api_core\_python_version_support.py:266: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [2]:
def load_reviews(path):
    try:
        df = pd.read_csv(path)
        assert "review_text" in df.columns
        return df
    except FileNotFoundError:
        raise FileNotFoundError(f"Dataset not found: {path}")
    except AssertionError:
        raise ValueError("Column review_text missing from dataset.")

df = load_reviews(REVIEWS_PATH)
print("Loaded:", df.shape)


Loaded: (10199, 20)


In [3]:
def clean_and_tokenize(text):
    if not isinstance(text, str):
        return []
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text.lower())
    return text.split()

sentences = df["review_text"].apply(clean_and_tokenize).tolist()
sentences = [s for s in sentences if len(s) > 2]
print("Sentences for training:", len(sentences))


Sentences for training: 8984


## (a) 'cheap' gets ONE vector — polysemy problem

In [4]:
def train_word2vec(sentences, window, vector_size=100, min_count=2, seed=42):
    model = Word2Vec(
        sentences=sentences,
        vector_size=vector_size,
        window=window,
        min_count=min_count,
        workers=4,
        seed=seed
    )
    return model

model_w5 = train_word2vec(sentences, window=5)
print("Vocab size:", len(model_w5.wv))


Vocab size: 253


In [6]:
def get_vector(model, word):
    if word in model.wv:
        return model.wv[word]
    return None

def cosine_between(model, word_a, word_b):
    va = get_vector(model, word_a)
    vb = get_vector(model, word_b)
    if va is None or vb is None:
        return None
    return cosine_similarity(va, vb)       # ← flat vectors, no [0][0]

sim_affordable = cosine_between(model_w5, TARGET_WORD, "affordable")   # ← model not model_w5
sim_flimsy     = cosine_between(model_w5, TARGET_WORD, "flimsy")

print(f"Vector dimension for 'cheap': {model_w5.wv[TARGET_WORD].shape}")

if sim_affordable is not None:
    print(f"cosine('cheap', 'affordable') = {sim_affordable:.4f}")
else:
    print("'affordable' not in vocabulary")

if sim_flimsy is not None:
    print(f"cosine('cheap', 'flimsy')     = {sim_flimsy:.4f}")
else:
    print("'flimsy' not in vocabulary")

print("Both senses share the SAME single vector — this is the polysemy problem.")

Vector dimension for 'cheap': (100,)
'affordable' not in vocabulary
'flimsy' not in vocabulary
Both senses share the SAME single vector — this is the polysemy problem.


## (b) Context-based disambiguation

In [ ]:
def get_anchor_vector(model, anchor_words):
    vecs = [model.wv[w] for w in anchor_words if w in model.wv]
    if not vecs:
        raise ValueError("None of the anchor words found in vocabulary.")
    return np.mean(vecs, axis=0)

def get_context_vector(model, sentence_tokens, target, window=3):
    if target not in sentence_tokens:
        return None
    idx = sentence_tokens.index(target)
    start = max(0, idx - window)
    end   = min(len(sentence_tokens), idx + window + 1)
    context = [t for t in sentence_tokens[start:end] if t != target and t in model.wv]
    if not context:
        return None
    return np.mean([model.wv[w] for w in context], axis=0)

def disambiguate_cheap(model, sentence, anchor_affordable, anchor_low_quality):
    tokens = clean_and_tokenize(sentence)
    ctx_vec = get_context_vector(model, tokens, TARGET_WORD)
    if ctx_vec is None:
        return "cannot disambiguate — no context words in vocab"
    anc_aff = get_anchor_vector(model, anchor_affordable)
    anc_low = get_anchor_vector(model, anchor_low_quality)
    sim_aff = float(cosine_similarity([ctx_vec], [anc_aff])[0][0])
    sim_low = float(cosine_similarity([ctx_vec], [anc_low])[0][0])
    label = "affordable" if sim_aff > sim_low else "low-quality"
    return {"label": label, "sim_affordable": round(sim_aff, 4), "sim_low_quality": round(sim_low, 4)}

test_sentences = [
    "This phone is cheap and great value for money",
    "The build quality is cheap and it broke after two days",
    "Surprisingly cheap for a premium brand product",
    "The plastic feels cheap and the buttons are wobbly",
]

for s in test_sentences:
    result = disambiguate_cheap(model_w5, s, ANCHOR_AFFORDABLE, ANCHOR_LOW_QUALITY)
    print(f"Sentence: {s}")
    print(f"  => {result}\n")


Sentence: This phone is cheap and great value for money
  => {'label': 'affordable', 'sim_affordable': 0.3892, 'sim_low_quality': 0.3303}

Sentence: The build quality is cheap and it broke after two days
  => {'label': 'low-quality', 'sim_affordable': 0.1603, 'sim_low_quality': 0.3688}

Sentence: Surprisingly cheap for a premium brand product
  => {'label': 'affordable', 'sim_affordable': 0.5438, 'sim_low_quality': 0.1697}

Sentence: The plastic feels cheap and the buttons are wobbly
  => {'label': 'low-quality', 'sim_affordable': 0.2877, 'sim_low_quality': 0.4099}



## (c) window_size=2 vs window_size=10

In [ ]:
model_w2  = train_word2vec(sentences, window=2)
model_w10 = train_word2vec(sentences, window=10)

def compare_nearest(model, word, topn=5):
    if word not in model.wv:
        return []
    return [(w, round(s, 4)) for w, s in model.wv.most_similar(word, topn=topn)]

print("Nearest to 'cheap' — window=2 (syntactic/local context):")
for w, s in compare_nearest(model_w2, TARGET_WORD):
    print(f"  {w:20s} {s}")

print("\nNearest to 'cheap' — window=10 (semantic/topic context):")
for w, s in compare_nearest(model_w10, TARGET_WORD):
    print(f"  {w:20s} {s}")

print("\nConclusion:")
print("  window=2  captures SYNTACTIC relationships (words adjacent in grammar).")
print("  window=10 captures SEMANTIC relationships (words in the same topic).")


Nearest to 'cheap' — window=2 (syntactic/local context):
  material             0.895
  price                0.8221
  thoda                0.7085
  worth                0.6244
  job                  0.5875

Nearest to 'cheap' — window=10 (semantic/topic context):
  material             0.9909
  finishing            0.9613
  much                 0.9074
  price                0.8204
  poor                 0.7897

Conclusion:
  window=2  captures SYNTACTIC relationships (words adjacent in grammar).
  window=10 captures SEMANTIC relationships (words in the same topic).
